# Neutral Atoms Environment Exploration

This notebook covers:
1. **Visualization** - board state, planned moves, gate interactions
2. **Random rollouts** - sanity check reward structure

In [1]:
import torch
import random
from neutral_atoms import (
    NeutralAtomsEnv, EnvConfig, GATE_ACTION, EMPTY_CELL,
    parallel_groups, count_groups, gates_to_moves
)

## 1. Visualization Helpers

In [2]:
def visualize_board(env, show_gates=True, show_moves=True):
    """Print board with atoms, planned moves, and current gate layer."""
    board = env.board
    h, w = board.shape
    positions = env.atom_positions
    
    # collect move arrows for current phase
    move_arrows = {}  # (row, col) -> list of (dst_row, dst_col)
    if show_moves and env.current_phase_moves:
        for move in env.current_phase_moves:
            sr, sc, dr, dc = move.tolist()
            if (sr, sc) not in move_arrows:
                move_arrows[(sr, sc)] = []
            move_arrows[(sr, sc)].append((dr, dc))
    
    # collect gate pairs for current layer
    gate_pairs = []
    if show_gates and env.tasks_done < len(env.tasks):
        gate_pairs = env.tasks[env.tasks_done]
    
    # header
    print(f"\n{'='*40}")
    print(f"Tasks: {env.tasks_done}/{len(env.tasks)} done | Cost: {env._cached_cost} | Entropy: {env._cached_entropy:.3f}")
    print(f"Actions: {len(env.actions)} | Phase moves: {len(env.current_phase_moves)}")
    print(f"{'='*40}")
    
    # column numbers
    print("    " + " ".join(f"{c:^3}" for c in range(w)))
    print("   +" + "---+" * w)
    
    for r in range(h):
        row_str = f"{r:2} |"
        for c in range(w):
            val = board[r, c].item()
            if val == EMPTY_CELL:
                cell = " . "
            else:
                cell = f" {val} "
            row_str += cell + "|"
        print(row_str)
        print("   +" + "---+" * w)
    
    # show current gate layer
    if gate_pairs:
        print(f"\nCurrent gates: {gate_pairs}")
        for q1, q2 in gate_pairs:
            p1, p2 = positions[q1].tolist(), positions[q2].tolist()
            print(f"  Gate({q1},{q2}): ({p1[0]},{p1[1]}) <-> ({p2[0]},{p2[1]})")
    
    # show planned moves
    if move_arrows:
        print(f"\nPlanned moves this phase:")
        for (sr, sc), dests in move_arrows.items():
            qubit = board[sr, sc].item()
            for dr, dc in dests:
                print(f"  Qubit {qubit}: ({sr},{sc}) -> ({dr},{dc})")

In [3]:
def visualize_groups(moves, label="Moves"):
    """Show how moves are grouped for parallel execution."""
    if len(moves) == 0:
        print(f"{label}: (empty)")
        return
    
    groups = parallel_groups(moves)
    n_groups = count_groups(moves)
    
    print(f"\n{label}: {len(moves)} moves -> {n_groups} parallel groups")
    for g in range(n_groups):
        mask = groups == g
        group_moves = moves[mask]
        print(f"  Group {g}: {len(group_moves)} moves")
        for m in group_moves:
            sr, sc, dr, dc = m.tolist()
            print(f"    ({sr},{sc}) -> ({dr},{dc})")

In [4]:
def action_to_string(action, env):
    """Convert action to human readable string."""
    if action == GATE_ACTION:
        return "GATE"
    from neutral_atoms.board import action_to_move
    move = action_to_move(action, env.atom_positions, env.board_shape)
    sr, sc, dr, dc = move.tolist()
    qubit = env.board[sr, sc].item()
    return f"Move Q{qubit}: ({sr},{sc})->({dr},{dc})"

## 2. Interactive Exploration

In [5]:
# create a simple environment
config = EnvConfig(
    board_height=4,
    board_width=4,
    num_qubits=4,
    budget=20,
    entropy_weight=0.0
)

# qubits in corners
initial_positions = [(0, 0), (0, 3), (3, 0), (3, 3)]

# two layers of gates
tasks = [
    [(0, 1), (2, 3)],  # layer 1: gate 0-1 and 2-3
    [(0, 2), (1, 3)],  # layer 2: gate 0-2 and 1-3
]

env = NeutralAtomsEnv(tasks, initial_positions, config)
env.reset()

visualize_board(env)


Tasks: 0/2 done | Cost: 4 | Entropy: 0.000
Actions: 0 | Phase moves: 0
     0   1   2   3 
   +---+---+---+---+
 0 | 0 | . | . | 1 |
   +---+---+---+---+
 1 | . | . | . | . |
   +---+---+---+---+
 2 | . | . | . | . |
   +---+---+---+---+
 3 | 2 | . | . | 3 |
   +---+---+---+---+

Current gates: [(0, 1), (2, 3)]
  Gate(0,1): (0,0) <-> (0,3)
  Gate(2,3): (3,0) <-> (3,3)


In [6]:
# show gate grouping for current layer
gate_moves = gates_to_moves(env.tasks[env.tasks_done], env.atom_positions)
visualize_groups(gate_moves, "Gate interactions")


Gate interactions: 2 moves -> 1 parallel groups
  Group 0: 2 moves
    (0,0) -> (0,3)
    (3,0) -> (3,3)


In [7]:
# show legal actions
legal = env.legal_actions()
print(f"Legal actions: {len(legal)}")
print("\nFirst 10 actions:")
for a in legal[:10]:
    print(f"  {a}: {action_to_string(a, env)}")

Legal actions: 49

First 10 actions:
  0: GATE
  2: Move Q0: (0,0)->(0,1)
  3: Move Q0: (0,0)->(0,2)
  5: Move Q0: (0,0)->(1,0)
  6: Move Q0: (0,0)->(1,1)
  7: Move Q0: (0,0)->(1,2)
  8: Move Q0: (0,0)->(1,3)
  9: Move Q0: (0,0)->(2,0)
  10: Move Q0: (0,0)->(2,1)
  11: Move Q0: (0,0)->(2,2)


In [11]:
# manually step through
# try moving qubit 1 closer to qubit 0

def find_move_action(env, qubit, dst_row, dst_col):
    """Find action that moves qubit to destination."""
    board_size = env.board_shape[0] * env.board_shape[1]
    flat_dest = dst_row * env.board_shape[1] + dst_col
    return 1 + qubit * board_size + flat_dest

# move qubit 1 from (0,3) to (0,1)
action = find_move_action(env, qubit=1, dst_row=0, dst_col=1)
print(f"Action: {action} = {action_to_string(action, env)}")

result = env.step(action)
print(f"Reward: {result.reward:.3f}")

visualize_board(env)

Action: 18 = Move Q1: (0,1)->(0,1)
Reward: -1.000

Tasks: 0/2 done | Cost: 12 | Entropy: 1.000
Actions: 4 | Phase moves: 4
     0   1   2   3 
   +---+---+---+---+
 0 | 0 | 1 | . | . |
   +---+---+---+---+
 1 | . | . | . | . |
   +---+---+---+---+
 2 | . | . | . | . |
   +---+---+---+---+
 3 | 2 | . | . | 3 |
   +---+---+---+---+

Current gates: [(0, 1), (2, 3)]
  Gate(0,1): (0,0) <-> (0,1)
  Gate(2,3): (3,0) <-> (3,3)

Planned moves this phase:
  Qubit -1: (0,3) -> (0,1)
  Qubit 1: (0,1) -> (0,1)
  Qubit 1: (0,1) -> (0,1)
  Qubit 1: (0,1) -> (0,1)


In [12]:
# now execute the gate layer
result = env.step(GATE_ACTION)
print(f"GATE executed! Reward: {result.reward:.3f}")

visualize_board(env)

GATE executed! Reward: 10.000

Tasks: 1/2 done | Cost: 2 | Entropy: 0.000
Actions: 5 | Phase moves: 0
     0   1   2   3 
   +---+---+---+---+
 0 | 0 | 1 | . | . |
   +---+---+---+---+
 1 | . | . | . | . |
   +---+---+---+---+
 2 | . | . | . | . |
   +---+---+---+---+
 3 | 2 | . | . | 3 |
   +---+---+---+---+

Current gates: [(0, 2), (1, 3)]
  Gate(0,2): (0,0) <-> (3,0)
  Gate(1,3): (0,1) <-> (3,3)


## 3. Random Rollout Baseline

In [13]:
def random_rollout(env, verbose=False):
    """Execute random actions until done. Returns total reward and action count."""
    env = env.clone()
    total_reward = 0
    actions_taken = []
    
    while True:
        legal = env.legal_actions()
        if not legal:
            break
        
        action = random.choice(legal)
        actions_taken.append(action)
        result = env.step(action)
        total_reward += result.reward
        
        if verbose:
            print(f"Action: {action_to_string(action, env)} -> reward: {result.reward:.3f}")
        
        if result.done:
            break
    
    return {
        'total_reward': total_reward,
        'final_cost': env._cached_cost,
        'actions': len(actions_taken),
        'tasks_done': env.tasks_done,
    }

In [15]:
# single verbose rollout
env_fresh = NeutralAtomsEnv(tasks, initial_positions, config)
env_fresh.reset()

print("=== Random Rollout (verbose) ===")
result = random_rollout(env_fresh, verbose=True)
print(f"\nFinal: reward={result['total_reward']:.3f}, cost={result['final_cost']}, actions={result['actions']}")

=== Random Rollout (verbose) ===
Action: Move Q3: (2,0)->(2,0) -> reward: -5.000
Action: GATE -> reward: 5.000
Action: Move Q3: (1,0)->(1,0) -> reward: -1.000
Action: Move Q2: (0,2)->(0,2) -> reward: 1.000
Action: Move Q1: (3,1)->(3,1) -> reward: -3.000
Action: Move Q0: (0,3)->(0,3) -> reward: -1.000
Action: Move Q2: (3,3)->(3,3) -> reward: -1.000
Action: Move Q1: (2,2)->(2,2) -> reward: 0.000
Action: Move Q0: (0,1)->(0,1) -> reward: -1.000
Action: Move Q1: (3,1)->(3,1) -> reward: -1.000
Action: Move Q3: (1,3)->(1,3) -> reward: 0.000
Action: Move Q0: (2,3)->(2,3) -> reward: -1.000
Action: Move Q0: (2,1)->(2,1) -> reward: 0.000
Action: Move Q3: (0,3)->(0,3) -> reward: 0.000
Action: Move Q3: (1,1)->(1,1) -> reward: -1.000
Action: Move Q1: (2,0)->(2,0) -> reward: 0.000
Action: Move Q1: (1,3)->(1,3) -> reward: -1.000
Action: Move Q0: (3,1)->(3,1) -> reward: -1.000
Action: Move Q0: (0,0)->(0,0) -> reward: -1.000
Action: Move Q1: (3,2)->(3,2) -> reward: -1.000

Final: reward=-13.000, cost=17

In [16]:
def run_rollout_stats(env, n_rollouts=100):
    """Run multiple rollouts and collect statistics."""
    results = [random_rollout(env) for _ in range(n_rollouts)]
    
    rewards = [r['total_reward'] for r in results]
    costs = [r['final_cost'] for r in results]
    actions = [r['actions'] for r in results]
    
    print(f"\n=== {n_rollouts} Random Rollouts ===")
    print(f"Reward: mean={sum(rewards)/len(rewards):.3f}, min={min(rewards):.3f}, max={max(rewards):.3f}")
    print(f"Final cost: mean={sum(costs)/len(costs):.2f}, min={min(costs)}, max={max(costs)}")
    print(f"Actions: mean={sum(actions)/len(actions):.1f}, min={min(actions)}, max={max(actions)}")
    
    return results

In [17]:
# run many rollouts
env_fresh = NeutralAtomsEnv(tasks, initial_positions, config)
env_fresh.reset()

stats = run_rollout_stats(env_fresh, n_rollouts=100)


=== 100 Random Rollouts ===
Reward: mean=-11.310, min=-21.000, max=4.000
Final cost: mean=15.31, min=0, max=25
Actions: mean=19.4, min=4, max=20


In [16]:
# compare: greedy "always GATE" vs random
def greedy_gate_rollout(env):
    """Always execute GATE immediately (no reconfig moves)."""
    env = env.clone()
    total_reward = 0
    
    while True:
        legal = env.legal_actions()
        if not legal:
            break
        
        # always pick GATE if available
        action = GATE_ACTION if GATE_ACTION in legal else legal[0]
        result = env.step(action)
        total_reward += result.reward
        
        if result.done:
            break
    
    return {'total_reward': total_reward, 'final_cost': env._cached_cost}

env_fresh = NeutralAtomsEnv(tasks, initial_positions, config)
env_fresh.reset()

greedy_result = greedy_gate_rollout(env_fresh)
print(f"\nGreedy GATE: reward={greedy_result['total_reward']:.3f}, final_cost={greedy_result['final_cost']}")
print(f"\nThis is the 'baseline' - no reconfig, just execute gates as-is.")
print(f"A good agent should beat this by making smart reconfig moves.")


Greedy GATE: reward=2.000, final_cost=0

This is the 'baseline' - no reconfig, just execute gates as-is.
A good agent should beat this by making smart reconfig moves.


## 4. Explore Different Scenarios

In [17]:
# scenario: atoms already well-positioned
print("=== Scenario: Well-positioned atoms ===")
config2 = EnvConfig(board_height=4, board_width=4, num_qubits=4, budget=20)
# place atoms so gates can parallelize
initial2 = [(0, 0), (0, 1), (1, 0), (1, 1)]  # 2x2 block
tasks2 = [[(0, 1), (2, 3)]]  # both gates can interact easily

env2 = NeutralAtomsEnv(tasks2, initial2, config2)
env2.reset()

visualize_board(env2)
print(f"\nCost lower bound: {env2.cost_lb}, upper bound: {env2.cost_ub}")

# greedy should do well here
greedy = greedy_gate_rollout(env2)
print(f"Greedy result: cost={greedy['final_cost']}")

=== Scenario: Well-positioned atoms ===

Tasks: 0/1 done | Cost: 1 | Entropy: 0.000
Actions: 0 | Phase moves: 0
     0   1   2   3 
   +---+---+---+---+
 0 | 0 | 1 | . | . |
   +---+---+---+---+
 1 | 2 | 3 | . | . |
   +---+---+---+---+
 2 | . | . | . | . |
   +---+---+---+---+
 3 | . | . | . | . |
   +---+---+---+---+

Current gates: [(0, 1), (2, 3)]
  Gate(0,1): (0,0) <-> (0,1)
  Gate(2,3): (1,0) <-> (1,1)

Cost lower bound: 1, upper bound: 2
Greedy result: cost=0


In [18]:
# scenario: atoms badly positioned
print("=== Scenario: Badly-positioned atoms ===")
config3 = EnvConfig(board_height=6, board_width=6, num_qubits=4, budget=30)
# place atoms in corners - far apart
initial3 = [(0, 0), (0, 5), (5, 0), (5, 5)]
# multiple layers
tasks3 = [
    [(0, 1), (2, 3)],
    [(0, 2), (1, 3)],
    [(0, 3), (1, 2)],
]

env3 = NeutralAtomsEnv(tasks3, initial3, config3)
env3.reset()

visualize_board(env3)
print(f"\nCost bounds: [{env3.cost_lb}, {env3.cost_ub}]")

# compare strategies
greedy = greedy_gate_rollout(env3)
print(f"\nGreedy: cost={greedy['final_cost']}, reward={greedy['total_reward']:.3f}")

random_stats = run_rollout_stats(env3, n_rollouts=50)

=== Scenario: Badly-positioned atoms ===

Tasks: 0/3 done | Cost: 4 | Entropy: 0.333
Actions: 0 | Phase moves: 0
     0   1   2   3   4   5 
   +---+---+---+---+---+---+
 0 | 0 | . | . | . | . | 1 |
   +---+---+---+---+---+---+
 1 | . | . | . | . | . | . |
   +---+---+---+---+---+---+
 2 | . | . | . | . | . | . |
   +---+---+---+---+---+---+
 3 | . | . | . | . | . | . |
   +---+---+---+---+---+---+
 4 | . | . | . | . | . | . |
   +---+---+---+---+---+---+
 5 | 2 | . | . | . | . | 3 |
   +---+---+---+---+---+---+

Current gates: [(0, 1), (2, 3)]
  Gate(0,1): (0,0) <-> (0,5)
  Gate(2,3): (5,0) <-> (5,5)

Cost bounds: [3, 6]

Greedy: cost=0, reward=4.000

=== 50 Random Rollouts ===
Reward: mean=-16.320, min=-22.000, max=-2.000
Final cost: mean=20.32, min=6, max=26
Actions: mean=30.0, min=30, max=30


In [19]:
# scenario: with entropy weight
print("=== Scenario: Entropy-weighted rewards ===")
config4 = EnvConfig(board_height=4, board_width=4, num_qubits=4, budget=20, entropy_weight=0.5)

env4 = NeutralAtomsEnv(tasks, initial_positions, config4)
env4.reset()

print(f"Initial cost: {env4._cached_cost}, entropy: {env4._cached_entropy:.3f}")

# compare with/without entropy
env_no_ent = NeutralAtomsEnv(tasks, initial_positions, 
    EnvConfig(board_height=4, board_width=4, num_qubits=4, budget=20, entropy_weight=0.0))
env_no_ent.reset()

print(f"\nRunning 50 rollouts each...")
stats_with_ent = run_rollout_stats(env4, 50)
stats_no_ent = run_rollout_stats(env_no_ent, 50)

=== Scenario: Entropy-weighted rewards ===
Initial cost: 2, entropy: 0.000

Running 50 rollouts each...

=== 50 Random Rollouts ===
Reward: mean=-12.192, min=-18.485, max=2.000
Final cost: mean=13.76, min=0, max=20
Actions: mean=19.7, min=4, max=20

=== 50 Random Rollouts ===
Reward: mean=-12.780, min=-19.000, max=2.000
Final cost: mean=14.78, min=0, max=21
Actions: mean=19.7, min=12, max=20


## 5. Inspect Parallel Grouping

In [20]:
# manually create moves and visualize grouping
from neutral_atoms import is_parallel_executable_batch

# 4 moves: some parallel, some not
test_moves = torch.tensor([
    [0, 0, 0, 3],  # move right along row 0
    [1, 0, 1, 3],  # move right along row 1 (parallel with above)
    [2, 0, 2, 3],  # move right along row 2 (parallel)
    [0, 1, 3, 1],  # move down along col 1 (crosses others!)
])

can_parallel = is_parallel_executable_batch(test_moves)
print("Can-parallel matrix (True = can run together):")
print(can_parallel.int())

visualize_groups(test_moves, "Test moves")

Can-parallel matrix (True = can run together):
tensor([[1, 1, 1, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 0],
        [0, 0, 0, 1]], dtype=torch.int32)

Test moves: 4 moves -> 2 parallel groups
  Group 0: 1 moves
    (0,1) -> (3,1)
  Group 1: 3 moves
    (0,0) -> (0,3)
    (1,0) -> (1,3)
    (2,0) -> (2,3)


In [21]:
# another example: all parallel
parallel_moves = torch.tensor([
    [0, 0, 0, 2],
    [1, 0, 1, 2],
    [2, 0, 2, 2],
    [3, 0, 3, 2],
])
visualize_groups(parallel_moves, "All parallel (same direction)")


All parallel (same direction): 4 moves -> 1 parallel groups
  Group 0: 4 moves
    (0,0) -> (0,2)
    (1,0) -> (1,2)
    (2,0) -> (2,2)
    (3,0) -> (3,2)


In [22]:
# worst case: all conflicting
conflicting_moves = torch.tensor([
    [0, 0, 3, 3],  # diagonal
    [0, 3, 3, 0],  # opposite diagonal
    [1, 0, 1, 3],  # horizontal
    [0, 1, 3, 1],  # vertical
])
visualize_groups(conflicting_moves, "All conflicting")


All conflicting: 4 moves -> 4 parallel groups
  Group 0: 1 moves
    (0,0) -> (3,3)
  Group 1: 1 moves
    (0,3) -> (3,0)
  Group 2: 1 moves
    (1,0) -> (1,3)
  Group 3: 1 moves
    (0,1) -> (3,1)
